In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-gcp/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 00 · The core idea — a durable agent loop in 60 lines

This notebook shows the *idea* before the engine. We use only the standard library to build a run store, a task queue and a worker. Then we break the loop with a crash and a duplicate delivery. Everything depends on the three invariants of the engine (`docs/primer.md` §2):

1. **Write the checkpoint before you enqueue the task.**
2. **`(run, step, attempt)` identifies the work. Any other task is stale.**
3. **Record side effects before the checkpoint, with the intent as the key.**

## The store, the queue and the run document

In [2]:
from collections import deque
import copy

RUNS = {}           # run_id -> run document (Firestore stands in for this)
EFFECTS = {}        # "run:intent" -> result (idempotent side effects)
QUEUE = deque()     # (run_id, step, attempt)   (Cloud Tasks)
SEEN_TASKS = set()  # Cloud Tasks rejects a second task with the same name

def enqueue(run_id, step, attempt):
    key = (run_id, step, attempt)
    if key in SEEN_TASKS:
        return False           # dedup by name
    SEEN_TASKS.add(key)
    QUEUE.append(key)
    return True

def start(run_id, first_step):
    RUNS[run_id] = {"step": first_step, "attempts": {first_step: 1}, "state": {}, "status": "RUNNING", "version": 0}
    enqueue(run_id, first_step, 1)

## Steps with an idempotent side effect

`effect()` runs `fn` at most one time for each `(run, intent)`. Look at the order: `effect()` records the effect *before* the caller writes the checkpoint.

In [3]:
def effect(run_id, intent, fn):
    key = f"{run_id}:{intent}"
    if key in EFFECTS:
        return EFFECTS[key]            # already happened (retry after a crash)
    result = fn()
    EFFECTS[key] = result
    return result

CHARGES = []
def reserve(run_id, state):
    state["reservation"] = "res-1"
    return "charge"                    # next step
def charge(run_id, state):
    rec = effect(run_id, "charge", lambda: CHARGES.append("charged") or {"payment_id": "pay-1"})
    state["payment"] = rec["payment_id"]
    return "ship"
def ship(run_id, state):
    state["shipment"] = "shp-1"
    return None                        # done

STEPS = {"reserve": reserve, "charge": charge, "ship": ship}

## The worker

The whole repository builds on this loop. Fill in the guard and the order of the operations.

In [4]:
CRASH_AFTER_COMMIT = {"on": False}

def worker(run_id, step, attempt):
    run = RUNS[run_id]
    # (2) stale guard: is this exactly the work the run expects right now?
    if run["status"] != "RUNNING" or run["step"] != step or run["attempts"][step] != attempt:
        return "stale"
    state = copy.deepcopy(run["state"])            # work on a copy; commit atomically below
    nxt = STEPS[step](run_id, state)
    # --- (1) checkpoint FIRST ---
    run["state"] = state
    run["version"] += 1
    if nxt is None:
        run["status"], run["step"] = "SUCCEEDED", None
        return "done"
    run["step"] = nxt
    run["attempts"][nxt] = run["attempts"].get(nxt, 0) + 1
    if CRASH_AFTER_COMMIT["on"]:
        CRASH_AFTER_COMMIT["on"] = False
        raise RuntimeError("worker died after the checkpoint, before enqueue")
    # --- then enqueue ---
    enqueue(run_id, nxt, run["attempts"][nxt])
    return "ok"

def drain():
    log = []
    while QUEUE:
        task = QUEUE.popleft()
        try:
            log.append((task[1], worker(*task)))
        except RuntimeError as e:
            log.append((task[1], f"CRASH: {e}"))
    return log

## Happy path

In [5]:
start("order-1", "reserve")
print(drain())
print(RUNS["order-1"]["status"], RUNS["order-1"]["state"], "charges:", CHARGES)
assert RUNS["order-1"]["status"] == "SUCCEEDED" and CHARGES == ["charged"]

[('reserve', 'ok'), ('charge', 'ok'), ('ship', 'done')]
SUCCEEDED {'reservation': 'res-1', 'payment': 'pay-1', 'shipment': 'shp-1'} charges: ['charged']


## Crash after the checkpoint, before the enqueue

The run is consistent (`step=ship`, attempt 1), but no task exists. Something must re-drive the run, and that is the job of the **reaper**. The checkpoint already moved on, so the task that the reaper enqueues again is the *only* task that can execute.

In [6]:
CHARGES.clear()
start("order-2", "reserve")
CRASH_AFTER_COMMIT["on"] = True
print(drain())
run = RUNS["order-2"]
print("after crash:", run["status"], run["step"], run["attempts"], "queue:", list(QUEUE))

def reaper():
    """Re-enqueue the *current* attempt of every RUNNING run. Dedup makes this safe to call often."""
    n = 0
    for run_id, r in RUNS.items():
        if r["status"] == "RUNNING":
            n += enqueue(run_id, r["step"], r["attempts"][r["step"]])
    return n

print("reaper re-enqueued:", reaper())
print(drain())
assert RUNS["order-2"]["status"] == "SUCCEEDED" and CHARGES == ["charged"], "exactly one charge"

[('reserve', 'CRASH: worker died after the checkpoint, before enqueue')]
after crash: RUNNING charge {'reserve': 1, 'charge': 1} queue: []
reaper re-enqueued: 1
[('charge', 'ok'), ('ship', 'done')]


## Duplicate delivery

Cloud Tasks (and every real queue) gives at-least-once delivery. Deliver the `charge` task two times. Then look at how the guard rejects the second delivery.

In [7]:
CHARGES.clear()
start("order-3", "reserve")
task = QUEUE.popleft(); print(task, worker(*task))            # reserve
task = QUEUE.popleft(); print(task, worker(*task))            # charge (attempt 1)
print("duplicate:", worker(*task))                            # same (run, step, attempt) again
assert worker(*task) == "stale" and CHARGES == ["charged"]
print(drain())

('order-3', 'reserve', 1) ok
('order-3', 'charge', 1) ok
duplicate: stale
[('ship', 'done')]


## What the real engine adds

* **Leases**, so that two replicas cannot run the same step at the same time (`Engine.execute_task`).
* **Retries** with backoff, through an increase of the attempt, so that the old task becomes stale.
* **`Wait`** for input from a person, **`FanOut`** into child runs, **compensation** for sagas, and **budgets**.
* **Optimistic concurrency** on `version`, in place of the single-threaded dict of this notebook.

Continue with `01_durable_execution`.